# 0. La caja negra: `src/toolkit/` aplicado sobre datos reales

Este notebook no genera resultados nuevos -- demuestra cada módulo del
toolkit reusable de forma standalone, sobre datos reales ya descargados por
los 4 dominios (`financial_bcch`, `mining_cochilco`, `agriculture_worldbank`,
`consulting_excel_dwh`). El punto es mostrar que las mismas funciones sirven
para cualquier dominio, sin conocer nada sobre él.


## `missing_data.interpolate_within_group`

Interpola gaps DENTRO de cada país, nunca a través de un cambio de país --
sobre el panel agrícola real (Banco Mundial).


In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src.toolkit.missing_data import interpolate_within_group, missingness_report

agri = pd.read_csv("../data/processed/agriculture/agriculture_panel_clean.csv")
missingness_report(agri)


,columna,n_nulos,pct_nulos,dtype
0,country_iso3,0,0.0,object
1,country_name,0,0.0,object
2,year,0,0.0,int64
3,cereal_yield_kg_ha,0,0.0,float64
4,arable_land_pct,0,0.0,float64
5,fertilizer_kg_ha,0,0.0,float64
6,agri_land_pct,0,0.0,float64
7,crop_production_index,0,0.0,float64
8,rural_pop_pct,0,0.0,float64
9,irrigated_land_pct,0,0.0,float64


## `outliers.fix_implausible_level_jumps`

El error real encontrado en la UF de `mindicador.cl` (2014-12-29/30): dos
días seguidos corruptos, con valores parecidos ENTRE SÍ, que una comparación
día-a-día no detecta -- se necesita una mediana móvil. Reproducido acá con
los valores reales encontrados.


In [2]:
from src.toolkit.outliers import fix_implausible_level_jumps

demo = pd.DataFrame({
    "fecha": pd.date_range("2014-12-24", periods=10),
    "uf": [24627.10, 24627.10, 24627.10, 24627.10, 608.15, 607.38, 24627.10, 24627.10, 24627.10, 24627.10],
})
fixed, n = fix_implausible_level_jumps(demo, "uf", sort_by="fecha", threshold=0.5)
print(f"{n} valores corregidos")
fixed


2 valores corregidos


,fecha,uf
0,2014-12-24,24627.1
1,2014-12-25,24627.1
2,2014-12-26,24627.1
3,2014-12-27,24627.1
4,2014-12-28,24627.1
5,2014-12-29,24627.1
6,2014-12-30,24627.1
7,2014-12-31,24627.1
8,2015-01-01,24627.1
9,2015-01-02,24627.1


## `excel_cleaning`: encabezados multi-fila y formato ancho -> largo

Sobre un extracto real del WDI (Banco Mundial) -- columnas de año 1960-2024
en una sola fila.


In [3]:
from src.toolkit.excel_cleaning import wide_years_to_long

wdi_wide = pd.read_csv("../data/raw/consulting/wdi_curated_wide.csv").head(3)
long_sample = wide_years_to_long(
    wdi_wide, id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
)
long_sample.dropna(subset=["valor"]).head(8)


,Country Name,Country Code,Indicator Name,Indicator Code,periodo,valor
2,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1960,186.132432
5,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1961,186.947182
8,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1962,197.408105
11,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1963,225.447007
14,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1964,209.005786
17,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1965,226.883067
20,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1966,240.962194
23,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1967,243.824367


## `text_cleaning`: parseo de moneda y unificación fuzzy de nombres

Ejemplos ilustrativos de la función (no son un hallazgo de dominio, solo
muestran su comportamiento con inputs mixtos, el tipo de mezcla real que
aparece en exports de sistemas distintos).


In [4]:
from src.toolkit.text_cleaning import parse_currency, unify_similar_names

for raw in ["$1,200.50", "1.200,50", "(20.00)", "1234 (p)", "N/A"]:
    print(f"{raw!r:15s} -> {parse_currency(raw)}")

names = pd.Series(["SQM S.A.", "SQM SA", "sqm s.a.", "Albemarle Corp"])
unify_similar_names(names, threshold=80)


'$1,200.50'     -> 1200.5
'1.200,50'      -> 1200.5
'(20.00)'       -> -20.0
'1234 (p)'      -> 1234.0
'N/A'           -> None


0          SQM S.A.
1          SQM S.A.
2          SQM S.A.
3    Albemarle Corp
dtype: object

## `encoding.zscore_scale` + `torch_trainer`

El mismo escalador y el mismo loop de entrenamiento (`train_with_early_stopping`,
piso de 100 épocas) que usan los 4 dominios -- acá sobre un slice pequeño y
real de features financieras, solo para mostrar la mecánica sin repetir el
modelo completo (ya está en `01_financial_bcch.ipynb`).


In [5]:
import torch
from torch import nn
from src.toolkit.encoding import zscore_scale
from src.toolkit.torch_trainer import train_with_early_stopping

fin_features = pd.read_csv("../data/processed/financial/financial_features.csv").head(400)
X_cols = ["dolar_log_return_lag1", "dolar_volatility_5d", "tpm"]
scaled, stats = zscore_scale(fin_features, X_cols)
X = torch.tensor(scaled[X_cols].values, dtype=torch.float32)
y = torch.tensor(fin_features["target_next_return"].values, dtype=torch.float32).view(-1, 1)

model = nn.Sequential(nn.Linear(3, 8), nn.LeakyReLU(0.1), nn.Linear(8, 1))
result = train_with_early_stopping(model, X[:300], y[:300], X[300:], y[300:], loss_fn=nn.MSELoss(), min_epochs=100, max_epochs=150)
print(f"épocas corridas: {result.epochs_run}, mejor época: {result.best_epoch}")


épocas corridas: 150, mejor época: 150


## Conclusión

Las mismas ~20 funciones de `src/toolkit/` (limpieza, outliers, texto, Excel,
encoding, visualización, entrenamiento) se reusan sin cambios en los 4
dominios -- lo único que cambia entre dominios es el dato de entrada real y
la interpretación del resultado, nunca la técnica.
